### Testing generation and comparing results for basic LLM, standard RAG, and this NIR pipeline

This notebook evaluates the following pipelines to determine the most effective approach for generating answers:
1. Basic LLM: uses a general world overview description as context and generates a narrative element based on the query and this text
2. Standard RAG: creates a vector database with text fragments and generates a narrative element based on the query and retrieved context
3. This NIR pipeline (version with a pre-stage plan generation): retrieves context from a graph and then, first, generates an answer plan based on the query and context; second, generates the final answer using the plan and context (based on the query)
4. This NIR pipeline (version without pre-stages): retrieves context from a graph and then generates an answer based on the query and context

**Metrics used:**

RAG efficiency and world consistency:
1. Faithfulness (RAGAS) – measures how factually consistent the generated answer is with the provided context. It evaluates whether the answer is fully grounded in the retrieved information and does not introduce unsupported or hallucinated statements. It is computed by comparing each claim in the generated answer against the retrieved context and checking whether it can be directly inferred from it. Higher values indicate that the model strictly follows the given context without adding external or fabricated information.
2. Answer Relevancy (RAGAS) – measures how relevant the generated answer is to the input question. It evaluates whether the response directly addresses the query without drifting into unrelated information. It is typically computed by comparing semantic similarity between the question and the generated answer. Higher scores indicate that the answer is well-aligned with the user’s intent.
3. Context Precision (RAGAS) – measures how relevant the retrieved context passages are to the question. It evaluates the proportion of useful retrieved information compared to all retrieved context. It is computed by checking which retrieved chunks are actually relevant for answering the query. Higher values indicate that the retrieval step returns mostly useful and non-noisy information.
4. Context Recall (RAGAS) – measures how well the retrieved context covers all the information needed to answer the question. It evaluates whether all necessary supporting facts are present in the retrieved context. It is computed by comparing the required information for a correct answer with the retrieved passages. Higher values indicate that the retrieval system successfully captures most or all relevant knowledge.
5. Answer Correctness (RAGAS) – measures the overall correctness of the generated answer compared to a reference (ground truth) answer. It evaluates both factual accuracy and semantic similarity to the expected response. It is typically computed using a combination of exact matching, semantic similarity (via embeddings), and factual consistency checks. Higher values indicate that the answer is not only relevant but also factually correct.
6. BERTScore (Generated Text vs World Description) – measures semantic similarity between the generated text and the world description. It evaluates how well the generated content aligns with the predefined world context in terms of meaning rather than exact wording. The metric is computed using contextual embeddings by matching tokens between the generated text and the world description and calculating precision, recall, and F1 over these matches. Higher values indicate stronger consistency of the generated output with the established world setting.
7. BERTScore (Generated Text vs Ground Truth) – measures semantic similarity between the generated text and the reference (ground truth) answer. It evaluates how closely the model’s output matches the expected correct response in meaning. The score is computed using contextual embeddings in the same way as above, by aligning tokens between generated and reference texts. Higher values indicate that the generated answer is closer in meaning to the ground truth, even if the wording differs.
8. World Consistency (LLM-based evaluation) – evaluates whether the generated text is consistent with the given world description using LLM as a judge. The model is prompted to assess if the generated narrative element could exist within the defined world rules, lore, and constraints (information based on world description), and outputs a continuous score between 0 and 1. Higher scores indicate that the generated text is coherent with the world setting and does not violate its established rules or context.

Text and generated narrative element quality:
1. Distinct-2 – measures lexical diversity of the generated text by computing the ratio of unique bigrams (2-grams) to the total number of bigrams. It evaluates how varied the text is in terms of local word sequences. Higher values indicate more diverse and less repetitive language, while lower values suggest repetitive or templated phrasing.
2. Repetition-2 – measures the level of repetition in the generated text by calculating how often bigrams (2-grams) are repeated within the output. It captures redundancy and looping patterns in phrasing. Lower values indicate more fluent and non-redundant text, while higher values suggest repetitive or overly formulaic generation.
3. MAUVE – measures the distributional similarity between the generated text and reference human-written text distributions. It evaluates how close the model’s output is to human-like text in a probabilistic embedding space. The metric compares clusters of token embeddings from both distributions and quantifies their divergence. Higher MAUVE scores indicate that generated text is more similar to human-written text in style and structure.
4. Self-BLEU – measures diversity within a set of generated texts by treating each generated sample as a hypothesis and the rest as references. It computes BLEU scores across generated outputs to evaluate how similar they are to each other. Lower values indicate higher diversity among generated samples, while higher values suggest that outputs are too similar or repetitive across generations.
5. Interestingness (LLM-as-a-judge) – evaluates how interesting, engaging, and creatively meaningful the generated text is using an LLM as a judge. The metric outputs a score from 0 to 1. It considers several factors: (1) whether the content appropriately reflects choice and agency when the task allows it, (2) how diverse and stylistically appropriate the text is, including whether it fits the tone, style, and world of the game without being overly dramatic or inconsistent, and (3) how creative and novel the idea is, including whether it provides fresh insights, new information about the world, or unique player experiences. Higher scores indicate more engaging, original, and well-aligned narrative content.

In [16]:
#imports

import os
import sys
import tqdm
import pandas as pd
import numpy as np
import logging
import warnings
import json
from typing import Any, Dict, List, Optional, Literal
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_community.vectorstores import FAISS
from pandas import json_normalize
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

#some important stuff setup

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(project_root)
sys.path.insert(0, project_root)

results_dir = os.path.join(project_root, "assets", "outputs", "test_results")

logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("faiss").setLevel(logging.WARNING)
warnings.filterwarnings("ignore")
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

#this nir imports

from nir.llm.manager import ModelManager
from nir.llm.providers import ModelConfig

from nir.tests.test_datasets import TEST_DATA_TEXT1_SHORT, TEST_DATA_TEXT2_SHORT, TEST_DATA_TEXT3_SHORT, TEST_DATA_LORE_DESCRIPTION
from nir.tests.evaluator import analyze_generation, compare_pipelines_directional, compute_effect_size, extend_dataset
from nir.tests.metrics import compute_mauve_en, compute_mauve_ru, compute_self_bleu

from nir.graph.graph_storages.networkx_graph import NetworkXGraph
from nir.core.context_retriever import form_context_with_llm
from nir.core.answers_generator import generate_plan
from nir.core.answers_generator import filter_context
from nir.core.answers_generator import generate_answer_based_on_plan
from nir.core.answers_generator import generate_answer_based_on_context

In [15]:
#models setup

manager = ModelManager()

instruct_model_config = ModelConfig(model_name="mistral:7b-instruct-q2_K", temperature=0.0)
instruct_llm = manager.create_chat_model(name="evaluation_model_tests", option="ollama", config=instruct_model_config)

answer_model_config = ModelConfig(model_name="llama3.2:latest", temperature=0.7)
answer_llm = manager.create_chat_model(name="generation_model_tests", option="ollama", config=answer_model_config)

embeddings_model = manager.create_embedding_model(name="embeddings_tests", option="ollama", model_name="evilfreelancer/enbeddrus:v0.2")

In [17]:
#data setup

test_data_lore_description = TEST_DATA_LORE_DESCRIPTION
test_data_design_document = TEST_DATA_TEXT2_SHORT
test_data_scenario = TEST_DATA_TEXT3_SHORT

**Testing basic LLM**

In [ ]:
def run_generation_tests_basic_llm(test_data: Dict[str, Any], dataset_name: str, output_filename: str, language: str="en") -> list[str]:
    all_metrics = []
    generated = []
    references = []

    for task in tqdm.tqdm(test_data["tasks"], desc=f"Testing basic llm generation on {dataset_name}"):
        query = task["query"]
        reference = task["reference"]
        category = task.get("category", "default")
        context = test_data.get("text_summary", "")
        if language == "ru":
            propmt = f"""
                Используй приведенный контекст и напиши ответ за запрос пользователя. 
                Следуй его инструкциям и напиши то, что пользователь от тебя ждет. \nКонтекст:\n{context}\nЗапрос\n{query}"
            """
        else:
            prompt = f"""
                Use provided context to answer user's query.
                Follow their insruction and write that user is expecting from you. \nContext:\n{context}\nQuery:\n{query}
            """
        answer_final = answer_llm.invoke(prompt)

        generated.append({"category": category, "generated_text": answer_final})
        references.append(reference)

        metrics = analyze_generation(
            generated_text=answer_final,
            context=context,
            lore_summary=context,
            reference_text=reference,
            query=query,
            category=category,
            evaluation_llm=instruct_llm,
            language="en"
        )

        all_metrics.append(metrics)

    generated_texts = [text["generated_text"] for text in generated]

    if language == "ru":
        mauve_score = compute_mauve_ru(generateds=generated_texts, references=references)
    else:
        mauve_score = compute_mauve_en(generateds=generated_texts, references=references)
    self_bleu_score = compute_self_bleu(generateds=generated_texts)

    print(f"RESULT FOR {dataset_name.capitalize()}")
    if not all_metrics:
        display(pd.DataFrame({"status": ["No data for analysis"]}))
    else:
        df = pd.DataFrame(all_metrics)
        num_cols = df.select_dtypes(include="number").columns.tolist()

        if "category" in df.columns and len(df) > 0:
            cat_df = df.groupby("category")[num_cols].mean().reset_index()
        else:
            cat_df = df[num_cols].mean().to_frame().T
            cat_df["category"] = "default"

        overall = {col: df[col].mean() for col in num_cols}
        overall["category"] = "OVERALL"
        overall_df = pd.DataFrame([overall])

        final_df = pd.concat([cat_df, overall_df], ignore_index=True)
        cols_order = ["category"] + sorted([c for c in final_df.columns if c != "category"])
        final_df = final_df[cols_order]
        
        def pretty_table(df):
            display(HTML("""
                <style>
                table {
                    width: 100%;
                    table-layout: fixed;
                    max-width: 1500px;
                }
                th {
                    word-break: break-word;
                    white-space: normal;
                }
                td {
                    word-break: break-word;
                    white-space: normal;
                }
                </style>
            """))
            display(df.style.hide(axis="index"))
        pretty_table(final_df)

    print(f"Mauve metric for texts generated on {dataset_name}: {mauve_score}")
    print(f"Self-BLEU metric for texts generated on {dataset_name}: {self_bleu_score}")

    if generated:
        pipeline_dir = os.path.join(results_dir, "Basic LLM")
        os.makedirs(pipeline_dir, exist_ok=True)

        generated_with_metrics = []
        for i, gen_item in enumerate(generated):
            metrics_item = all_metrics[i] if i < len(all_metrics) else {}
            combined_entry = {
                "generated_text": gen_item["generated_text"],
                "category": gen_item["category"],
                "reference_text": references[i] if i < len(references) else None,
                "metrics": metrics_item
            }
            generated_with_metrics.append(combined_entry)

        output_json_path = os.path.join(pipeline_dir, output_filename)
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(generated_with_metrics, f, ensure_ascii=False, indent=2)

        print(f"Generated texts are saved here: {output_json_path}")

    return generated_texts

In [23]:
# texts_lore = run_generation_tests_basic_llm(test_data_lore_description, "lore description", "lore_description.json")
texts_design = run_generation_tests_basic_llm(test_data_design_document, "design document", "design_document.json")
# texts_scenario = run_generation_tests_basic_llm(test_data_scenario, "scenario", "scenario.json")

#all_generated_texts = texts_lore #+ texts_design + texts_scenario
#self_bleu_all = compute_self_bleu(generateds=all_generated_texts)
#print(f"Self-BLEU on ALL generated texts in english: {self_bleu_all}")

#texts_russian = run_generation_tests_basic_llm(test_data_scenario, "scenario in russian", "scenario_russian.json", "ru")

Testing basic llm generation on design document:   0%|          | 0/1 [00:00<?, ?it/s]Exception raised in Job[0]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Testing basic llm generation on design document: 100%|██████████| 1/1 [08:49<00:00, 529.53s/it]


Featurizing p:   0%|          | 0/1 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/1 [00:00<?, ?it/s]

RESULT FOR Design document


category,answer_correctness,answer_relevancy,bert_score_reference,bert_score_source,context_precision,context_recall,distinct_2,faithfulness,interestingness,repetition_2,semantic_similarity,world_consistency
quest,nan,0.803978,0.771356,0.799861,0.000000,0.000000,0.896887,nan,0.850000,0.081712,0.902397,0.800000
OVERALL,nan,0.803978,0.771356,0.799861,0.000000,0.000000,0.896887,nan,0.850000,0.081712,0.902397,0.800000


Mauve metric for texts generated on design document: 0.16724536689815506
Self-BLEU metric for texts generated on design document: 0.0
Generated texts are saved here: e:\YourLittleNarrativeAdviser\assets\outputs\test_results\Basic LLM\design_document.json


**Testing standard RAG**

In [4]:
def run_generation_tests_standart_rag(test_data: Dict[str, Any], dataset_name: str, output_filename: str, language: str="en") -> list[str]:
    all_metrics = []
    generated = []
    references = []

    filepath = test_data["path_to_text"]
    loader = TextLoader(filepath, encoding="utf-8")
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    chunks: List[Document] = text_splitter.split_documents(documents)
    vectorstore = FAISS.from_documents(chunks, embeddings_model)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    for task in tqdm.tqdm(test_data["tasks"], desc=f"Testing standard RAG generation on {dataset_name}"):
        query = task["query"]
        reference = task["reference"]
        category = task.get("category", "default")

        docs = retriever.invoke(query)
        context = "\n\n".join(doc.page_content for doc in docs)
        
        if language == "ru":
            prompt = f"""
                Используй приведенный контекст и напиши ответ за запрос пользователя. 
                Следуй его инструкциям и напиши то, что пользователь от тебя ждет. \nКонтекст:\n{context}\nЗапрос\n{query}"
            """
        else:
            prompt = f"""
                Use provided context to answer user's query.
                Follow their insruction and write that user is expecting from you. \nContext:\n{context}\nQuery:\n{query}
            """
        answer_final = answer_llm.invoke(prompt)

        generated.append({"category": category, "generated_text": answer_final})
        references.append(reference)

        metrics = analyze_generation(
            generated_text=answer_final,
            context=context,
            lore_summary=context,
            reference_text=reference,
            query=query,
            category=category,
            evaluation_llm=instruct_llm,
            language="en"
        )

        all_metrics.append(metrics)

    generated_texts = [text["generated_text"] for text in generated]

    if language == "ru":
        mauve_score = compute_mauve_ru(generateds=generated_texts, references=references)
    else:
        mauve_score = compute_mauve_en(generateds=generated_texts, references=references)
    self_bleu_score = compute_self_bleu(generateds=generated_texts)

    print(f"RESULT FOR {dataset_name.capitalize()}")
    if not all_metrics:
        display(pd.DataFrame({"status": ["No data for analysis"]}))
    else:
        df = pd.DataFrame(all_metrics)
        num_cols = df.select_dtypes(include="number").columns.tolist()

        if "category" in df.columns and len(df) > 0:
            cat_df = df.groupby("category")[num_cols].mean().reset_index()
        else:
            cat_df = df[num_cols].mean().to_frame().T
            cat_df["category"] = "default"

        overall = {col: df[col].mean() for col in num_cols}
        overall["category"] = "OVERALL"
        overall_df = pd.DataFrame([overall])

        final_df = pd.concat([cat_df, overall_df], ignore_index=True)
        cols_order = ["category"] + sorted([c for c in final_df.columns if c != "category"])
        final_df = final_df[cols_order]

        display(final_df.style.format(precision=4))

    print(f"Mauve metric for texts generated on {dataset_name}: {mauve_score}")
    print(f"Self-BLEU metric for texts generated on {dataset_name}: {self_bleu_score}")

    if generated:
        pipeline_dir = os.path.join(results_dir, "Standard RAG")
        os.makedirs(pipeline_dir, exist_ok=True)

        generated_with_metrics = []
        for i, gen_item in enumerate(generated):
            metrics_item = all_metrics[i] if i < len(all_metrics) else {}
            combined_entry = {
                "generated_text": gen_item["generated_text"],
                "category": gen_item["category"],
                "reference_text": references[i] if i < len(references) else None,
                "metrics": metrics_item
            }
            generated_with_metrics.append(combined_entry)

        output_json_path = os.path.join(pipeline_dir, output_filename)
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(generated_with_metrics, f, ensure_ascii=False, indent=2)

        print(f"Generated texts are saved here: {output_json_path}")

    return generated_texts

In [5]:
texts_lore = run_generation_tests_standart_rag(test_data_lore_description, "lore description", "lore_description.json")
# texts_design = run_generation_tests_standart_rag(test_data_design_document, "design document", "design_document.json")
# texts_scenario = run_generation_tests_standart_rag(test_data_scenario, "scenario", "scenario.json")

all_generated_texts = texts_lore #+ texts_design + texts_scenario
self_bleu_all = compute_self_bleu(generateds=all_generated_texts)
print(f"Self-BLEU on ALL generated texts in english: {self_bleu_all}")

#texts_russian = run_generation_tests_standart_rag(test_data_scenario, "scenario in russian", "scenario_russian.json", "ru")

Testing standard RAG generation on lore description:   0%|          | 0/25 [00:00<?, ?it/s]04/29/2026 13:33:26 - ERROR - 	 Exception raised in Job[0]: TimeoutError()
04/29/2026 13:37:18 - ERROR - 	 Exception raised in Job[5]: TimeoutError()
Testing standard RAG generation on lore description:   8%|▊         | 2/25 [12:16<2:17:52, 359.67s/it]04/29/2026 13:45:43 - ERROR - 	 Exception raised in Job[0]: TimeoutError()
04/29/2026 13:49:42 - ERROR - 	 Exception raised in Job[5]: TimeoutError()
Testing standard RAG generation on lore description:  12%|█▏        | 3/25 [19:19<2:22:29, 388.63s/it]04/29/2026 13:56:07 - ERROR - 	 Exception raised in Job[5]: TimeoutError()
Testing standard RAG generation on lore description:  20%|██        | 5/25 [32:01<2:08:12, 384.61s/it]04/29/2026 14:05:27 - ERROR - 	 Exception raised in Job[0]: TimeoutError()
04/29/2026 14:09:30 - ERROR - 	 Exception raised in Job[4]: ResponseError(the input length exceeds the context length (status code: 400))
04/29/2026 14:1

Featurizing p:   0%|          | 0/25 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/25 [00:00<?, ?it/s]

RESULT FOR Lore description


,category,answer_correctness,answer_relevancy,bert_score_reference,bert_score_source,context_precision,context_recall,distinct_2,faithfulness,interestingness,repetition_2,semantic_similarity,world_consistency
0,character descripion,0.6160,0.5579,0.8026,0.8108,1.0000,1.0000,0.9408,1.0000,0.7500,0.0503,0.9641,0.9500
1,character description,0.4011,0.5149,0.8155,0.8078,0.9198,1.0000,0.9147,1.0000,0.7375,0.0572,0.9424,0.8875
2,dialogue,0.4043,0.2559,0.8019,0.8139,0.4508,0.1833,0.9477,0.2000,0.7780,0.0490,0.8774,0.7100
3,item description,0.5021,0.5964,0.8079,0.8093,0.7400,0.2000,0.9705,0.8000,0.6740,0.0249,0.9170,0.6600
4,location description,0.2303,0.6208,0.8115,0.8067,0.5944,1.0000,0.9164,1.0000,0.7900,0.0542,0.9312,0.7900
5,quest,0.4245,0.5535,0.7818,0.7991,0.3000,0.8000,0.8340,0.9630,0.6900,0.0943,0.9481,0.6800
6,OVERALL,0.4237,0.5100,0.8032,0.8075,0.6042,0.6215,0.9177,0.6993,0.7344,0.0557,0.9183,0.7480


Mauve metric for texts generated on lore description: 0.22534903230402542
Self-BLEU metric for texts generated on lore description: 0.11290011453097142
Generated texts are saved here: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\Standard RAG\lore_description.json
Self-BLEU on ALL generated texts in english: 0.11290011453097142


**Testing ThisNIRPipeline with two-staged generation**

In [4]:
def run_generation_tests_ThisNIRPipeline_two_stages(test_data: Dict[str, Any], dataset_name: str, output_filename: str, language: str="en") -> list[str]:
    all_metrics = []
    generated = []
    references = []

    filepath = test_data["path_to_graph"]
    graph = NetworkXGraph()
    graph.load(filepath)

    for task in tqdm.tqdm(test_data["tasks"], desc=f"Testing This NIR two-staged generation on {dataset_name}"):
        query = task["query"]
        reference = task["reference"]
        category = task.get("category", "default")

        this_graph_embeddings = manager.get_embedding_model(graph.get_embedding_model())

        context = form_context_with_llm(query, graph, instruct_llm, this_graph_embeddings, language)
        answer_plan = generate_plan(query, context, answer_llm, False, language)
        answer_final = generate_answer_based_on_plan(query, answer_plan, context, answer_llm, language)

        generated.append({"category": category, "generated_text": answer_final})
        references.append(reference)

        metrics = analyze_generation(
            generated_text=answer_final,
            context=context,
            lore_summary=context,
            reference_text=reference,
            query=query,
            category=category,
            evaluation_llm=instruct_llm,
            language="en"
        )

        all_metrics.append(metrics)

    generated_texts = [text["generated_text"] for text in generated]

    if language == "ru":
        mauve_score = compute_mauve_ru(generateds=generated_texts, references=references)
    else:
        mauve_score = compute_mauve_en(generateds=generated_texts, references=references)
    self_bleu_score = compute_self_bleu(generateds=generated_texts)

    print(f"RESULT FOR {dataset_name.capitalize()}")
    if not all_metrics:
        display(pd.DataFrame({"status": ["No data for analysis"]}))
    else:
        df = pd.DataFrame(all_metrics)
        num_cols = df.select_dtypes(include="number").columns.tolist()

        if "category" in df.columns and len(df) > 0:
            cat_df = df.groupby("category")[num_cols].mean().reset_index()
        else:
            cat_df = df[num_cols].mean().to_frame().T
            cat_df["category"] = "default"

        overall = {col: df[col].mean() for col in num_cols}
        overall["category"] = "OVERALL"
        overall_df = pd.DataFrame([overall])

        final_df = pd.concat([cat_df, overall_df], ignore_index=True)
        cols_order = ["category"] + sorted([c for c in final_df.columns if c != "category"])
        final_df = final_df[cols_order]

        display(final_df.style.format(precision=4))

    print(f"Mauve metric for texts generated on {dataset_name}: {mauve_score}")
    print(f"Self-BLEU metric for texts generated on {dataset_name}: {self_bleu_score}")

    if generated:
        pipeline_dir = os.path.join(results_dir, "This NIR (two-staged generation)")
        os.makedirs(pipeline_dir, exist_ok=True)

        generated_with_metrics = []
        for i, gen_item in enumerate(generated):
            metrics_item = all_metrics[i] if i < len(all_metrics) else {}
            combined_entry = {
                "generated_text": gen_item["generated_text"],
                "category": gen_item["category"],
                "reference_text": references[i] if i < len(references) else None,
                "metrics": metrics_item
            }
            generated_with_metrics.append(combined_entry)

        output_json_path = os.path.join(pipeline_dir, output_filename)
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(generated_with_metrics, f, ensure_ascii=False, indent=2)

        print(f"Generated texts are saved here: {output_json_path}")

    return generated_texts

In [5]:
texts_lore = run_generation_tests_ThisNIRPipeline_two_stages(test_data_lore_description, "lore description", "lore_description.json")
#texts_design = run_generation_tests_ThisNIRPipeline_two_stages(test_data_design_document, "design document", "design_document.json")
#texts_scenario = run_generation_tests_ThisNIRPipeline_two_stages(test_data_scenario, "scenario", "scenario.json")

all_generated_texts = texts_lore #+ texts_design + texts_scenario
self_bleu_all = compute_self_bleu(generateds=all_generated_texts)
print(f"Self-BLEU on ALL generated texts in english: {self_bleu_all}")

#texts_russian = run_generation_tests_ThisNIRPipeline_two_stages(test_data_scenario, "scenario in russian", "scenario_russian.json", "ru")

04/29/2026 18:55:39 - INFO - 	 Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
Testing This NIR two-staged generation on lore description:   0%|          | 0/25 [00:00<?, ?it/s]04/29/2026 18:59:14 - ERROR - 	 Exception raised in Job[2]: TimeoutError()
Testing This NIR two-staged generation on lore description:   4%|▍         | 1/25 [05:43<2:17:16, 343.20s/it]04/29/2026 19:05:19 - ERROR - 	 Exception raised in Job[2]: TimeoutError()
Testing This NIR two-staged generation on lore description:   8%|▊         | 2/25 [13:46<2:43:09, 425.65s/it]04/29/2026 19:12:16 - ERROR - 	 Exception raised in Job[0]: TimeoutError()
04/29/2026 19:14:47 - ERROR - 	 Exception raised in Job[2]: TimeoutError()
04/29/2026 19:17:19 - ERROR - 	 Exception raised in Job[4]: TimeoutError()
04/29/2026 19:19:19 - ERROR - 	 Exception raised in Job[5]: TimeoutError()
Testing This NIR two-staged generation on lore description:  12%|█▏        | 3/25 [24:31<3

Featurizing p:   0%|          | 0/25 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/25 [00:00<?, ?it/s]

RESULT FOR Lore description


,category,answer_correctness,answer_relevancy,bert_score_reference,bert_score_source,context_precision,context_recall,distinct_2,faithfulness,interestingness,repetition_2,semantic_similarity,world_consistency
0,character descripion,0.6119,0.5367,0.7963,0.8103,nan,1.0000,0.9355,0.0000,0.8500,0.0466,0.9476,0.1500
1,character description,0.6051,0.3257,0.8083,0.8026,1.0000,0.8438,0.9259,1.0000,0.8000,0.0531,0.9205,0.7625
2,dialogue,0.2957,0.3759,0.8070,0.8077,0.1750,0.2800,0.9674,0.4750,0.8100,0.0290,0.8828,0.7814
3,item description,0.4690,0.6896,0.8175,0.8003,0.7374,0.5833,0.9769,0.9600,0.6100,0.0209,0.8932,0.9200
4,location description,0.4427,0.6692,0.8108,0.8122,0.4667,0.9750,0.8962,1.0000,0.7100,0.0741,0.9270,0.8300
5,quest,0.2383,0.6461,0.7789,0.8088,nan,0.7500,0.8257,0.3667,0.7400,0.0985,0.9531,0.7300
6,OVERALL,0.4087,0.5497,0.8040,0.8066,0.6072,0.6877,0.9188,0.6504,0.7360,0.0549,0.9079,0.7803


Mauve metric for texts generated on lore description: 0.22534903230402542
Self-BLEU metric for texts generated on lore description: 0.13316674198418377
Generated texts are saved here: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\This NIR (two-staged generation)\lore_description.json
Self-BLEU on ALL generated texts in english: 0.13316674198418377


**Testing ThisNIRPipeline with one-staged generation**

In [4]:
def run_generation_tests_ThisNIRPipeline_one_stage(test_data: Dict[str, Any], dataset_name: str, output_filename: str, language: str="en") -> list[str]:
    all_metrics = []
    generated = []
    references = []

    filepath = test_data["path_to_graph"]
    graph = NetworkXGraph()
    graph.load(filepath)

    for task in tqdm.tqdm(test_data["tasks"], desc=f"Testing This NIR one-stage generation on {dataset_name}"):
        query = task["query"]
        reference = task["reference"]
        category = task.get("category", "default")

        this_graph_embeddings = manager.get_embedding_model(graph.get_embedding_model())

        context = form_context_with_llm(query, graph, instruct_llm, this_graph_embeddings, language)
        answer_final = generate_answer_based_on_context(query, context, answer_llm, language)

        generated.append({"category": category, "generated_text": answer_final})
        references.append(reference)

        metrics = analyze_generation(
            generated_text=answer_final,
            context=context,
            lore_summary=context,
            reference_text=reference,
            query=query,
            category=category,
            evaluation_llm=instruct_llm,
            language="en"
        )

        all_metrics.append(metrics)

    generated_texts = [text["generated_text"] for text in generated]

    if language == "ru":
        mauve_score = compute_mauve_ru(generateds=generated_texts, references=references)
    else:
        mauve_score = compute_mauve_en(generateds=generated_texts, references=references)
    self_bleu_score = compute_self_bleu(generateds=generated_texts)

    print(f"RESULT FOR {dataset_name.capitalize()}")
    if not all_metrics:
        display(pd.DataFrame({"status": ["No data for analysis"]}))
    else:
        df = pd.DataFrame(all_metrics)
        num_cols = df.select_dtypes(include="number").columns.tolist()

        if "category" in df.columns and len(df) > 0:
            cat_df = df.groupby("category")[num_cols].mean().reset_index()
        else:
            cat_df = df[num_cols].mean().to_frame().T
            cat_df["category"] = "default"

        overall = {col: df[col].mean() for col in num_cols}
        overall["category"] = "OVERALL"
        overall_df = pd.DataFrame([overall])

        final_df = pd.concat([cat_df, overall_df], ignore_index=True)
        cols_order = ["category"] + sorted([c for c in final_df.columns if c != "category"])
        final_df = final_df[cols_order]

        display(final_df.style.format(precision=4))

    print(f"Mauve metric for texts generated on {dataset_name}: {mauve_score}")
    print(f"Self-BLEU metric for texts generated on {dataset_name}: {self_bleu_score}")

    if generated:
        pipeline_dir = os.path.join(results_dir, "This NIR (one-staged generation)")
        os.makedirs(pipeline_dir, exist_ok=True)

        generated_with_metrics = []
        for i, gen_item in enumerate(generated):
            metrics_item = all_metrics[i] if i < len(all_metrics) else {}
            combined_entry = {
                "generated_text": gen_item["generated_text"],
                "category": gen_item["category"],
                "reference_text": references[i] if i < len(references) else None,
                "metrics": metrics_item
            }
            generated_with_metrics.append(combined_entry)

        output_json_path = os.path.join(pipeline_dir, output_filename)
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(generated_with_metrics, f, ensure_ascii=False, indent=2)

        print(f"Generated texts are saved here: {output_json_path}")

    return generated_texts

In [5]:
texts_lore = run_generation_tests_ThisNIRPipeline_one_stage(test_data_lore_description, "lore description", "lore_description.json")
# texts_design = run_generation_tests_ThisNIRPipeline_one_stage(test_data_design_document, "design document", "design_document.json")
#texts_scenario = run_generation_tests_ThisNIRPipeline_one_stage(test_data_scenario, "scenario", "scenario.json")

all_generated_texts = texts_lore #+ texts_design + texts_scenario
self_bleu_all = compute_self_bleu(generateds=all_generated_texts)
print(f"Self-BLEU on ALL generated texts in english: {self_bleu_all}")

#texts_russian = run_generation_tests_ThisNIRPipeline_one_stage(test_data_scenario, "scenario in russian", "scenario_russian.json", "ru")

04/29/2026 22:53:56 - INFO - 	 Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
Testing This NIR one-stage generation on lore description:   0%|          | 0/25 [00:00<?, ?it/s]04/29/2026 22:57:23 - ERROR - 	 Exception raised in Job[2]: TimeoutError()
Testing This NIR one-stage generation on lore description:   4%|▍         | 1/25 [06:03<2:25:28, 363.69s/it]04/29/2026 23:03:38 - ERROR - 	 Exception raised in Job[2]: TimeoutError()
Testing This NIR one-stage generation on lore description:   8%|▊         | 2/25 [13:24<2:36:43, 408.85s/it]04/29/2026 23:11:05 - ERROR - 	 Exception raised in Job[2]: TimeoutError()
Testing This NIR one-stage generation on lore description:  20%|██        | 5/25 [32:53<2:10:59, 392.97s/it]04/29/2026 23:29:26 - ERROR - 	 Exception raised in Job[0]: TimeoutError()
04/29/2026 23:31:57 - ERROR - 	 Exception raised in Job[2]: TimeoutError()
04/29/2026 23:34:58 - ERROR - 	 Exception raised in Job[5]: 

Featurizing p:   0%|          | 0/25 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/25 [00:00<?, ?it/s]

RESULT FOR Lore description


,category,answer_correctness,answer_relevancy,bert_score_reference,bert_score_source,context_precision,context_recall,distinct_2,faithfulness,interestingness,repetition_2,semantic_similarity,world_consistency
0,character descripion,0.5407,0.6002,0.8022,0.8283,nan,1.0000,0.8631,1.0000,0.7500,0.0732,0.9627,0.3000
1,character description,0.4490,0.6689,0.8189,0.8137,1.0000,0.8688,0.9485,0.4375,0.7625,0.0422,0.9418,0.7250
2,dialogue,0.2917,0.3908,0.8210,0.8020,0.1750,0.2571,0.9565,0.7083,0.7900,0.0364,0.8835,0.8000
3,item description,0.4815,0.6806,0.8341,0.8045,0.6835,0.4333,0.9532,1.0000,0.7900,0.0376,0.9111,0.7100
4,location description,0.3709,0.6413,0.8249,0.8138,0.6817,0.7667,0.9042,0.6667,0.7600,0.0684,0.9359,0.8600
5,quest,nan,0.6453,0.8076,0.7976,nan,0.6667,0.8039,nan,0.7400,0.1110,0.9499,0.7250
6,OVERALL,0.4143,0.6026,0.8206,0.8069,0.5941,0.6011,0.9098,0.7402,0.7680,0.0603,0.9198,0.7470


Mauve metric for texts generated on lore description: 0.22534903230402542
Self-BLEU metric for texts generated on lore description: 0.12761417808254832
Generated texts are saved here: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\This NIR (one-staged generation)\lore_description.json
Self-BLEU on ALL generated texts in english: 0.12761417808254832


**Saving jsons as csv for future analysis**

In [2]:
def json_to_csv(json_path: str, csv_path: str = None, delimiter: str = ';') -> None:
    if csv_path is None:
        csv_path = json_path.replace('.json', '.csv')
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if not data:
        print("JSON is empty.")
        return

    df = json_normalize(
        data, 
        sep='_'
    )
    main_cols = ['category', 'reference_text', 'generated_text'] 
    
    existing_main_cols = [col for col in main_cols if col in df.columns]
    metric_cols = [col for col in df.columns if col.startswith('metrics_')]
    other_cols = [col for col in df.columns if col not in existing_main_cols and not col.startswith('metrics_')]
    
    final_order = existing_main_cols + metric_cols + other_cols
    df = df[final_order]
    df.to_csv(csv_path, index=False, encoding='utf-8-sig', sep=delimiter)
    
    print(f"JSON converted to: {csv_path}")
    return df

**Analyze results**

In [3]:
#diagram drawing
def plot_vertical_density_comparison(
    data_dict: Dict[str, pd.Series],
    metric_name: str,
    palette: Optional[List[str]] = None,
    figsize: tuple = (10, 8),
    show_median: bool = True,
    show_mean: bool = False,
    bw_adjust: float = 1.0,
    alpha_fill: float = 0.7,
    output_path: Optional[str] = None
) -> plt.Figure:

    pipelines = list(data_dict.keys())
    if palette is None:
        palette = sns.color_palette("muted", len(pipelines))
    fig, ax = plt.subplots(figsize=figsize)
    
    all_values = [vals.dropna().values for vals in data_dict.values()]
    global_min = min(np.min(v) for v in all_values if len(v) > 0)
    global_max = max(np.max(v) for v in all_values if len(v) > 0)
    x_margin = (global_max - global_min) * 0.05
    x_limits = (global_min - x_margin, global_max + x_margin)

    for idx, (pipeline, values) in enumerate(data_dict.items()):
        clean_vals = values.dropna()
        if len(clean_vals) == 0:
            continue
        from scipy.stats import gaussian_kde
        kde = gaussian_kde(clean_vals, bw_method=bw_adjust / np.std(clean_vals) if np.std(clean_vals) > 0 else 1.0)
        x_grid = np.linspace(x_limits[0], x_limits[1], 200)
        density = kde(x_grid)
        y_base = idx
        y_offset = 0.4

        ax.fill_betweenx(
            y=np.linspace(y_base - y_offset/2, y_base + y_offset/2, len(density)),
            x1=x_limits[0], 
            x2=x_grid,
            color=palette[idx % len(palette)],
            alpha=alpha_fill,
            label=pipeline
        )

        ax.plot(
            x_grid, 
            np.linspace(y_base - y_offset/2, y_base + y_offset/2, len(density)),
            color=palette[idx % len(palette)], 
            linewidth=1.5
        )

        if show_median:
            median_val = np.median(clean_vals)
            ax.plot([median_val, median_val], 
                   [y_base - y_offset/3, y_base + y_offset/3], 
                   color='black', linewidth=2, zorder=5)
            ax.text(median_val + (x_limits[1]-x_limits[0])*0.01, y_base, 
                   f'{median_val:.3f}', va='center', fontsize=9, fontweight='bold')

        if show_mean:
            mean_val = np.mean(clean_vals)
            ax.scatter([mean_val], [y_base], color='white', edgecolor='black', 
                      s=40, zorder=6, marker='o', label=f'{pipeline} mean')

    ax.set_xlabel(metric_name, fontsize=11)
    ax.set_yticks(range(len(pipelines)))
    ax.set_yticklabels(pipelines, fontsize=10)
    ax.set_xlim(x_limits)
    ax.set_ylim(-0.5, len(pipelines) - 0.5)

    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.tick_params(left=False)

    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"Graph is saved here: {output_path}")
    
    return fig

In [4]:
#evaluate tests and show comparison report
def run_pipeline_comparison_report(
    df_baseline: pd.DataFrame,
    df_proposed: pd.DataFrame,
    metrics: List[str],
    alternative: Literal['greater', 'less'] = 'greater',
    alpha: float = 0.05,
    effect_type: Literal['cohens_d', 'rank_biserial', 'auto'] = 'auto',
    baseline_name: str = "Baseline",
    proposed_name: str = "Proposed"
) -> pd.DataFrame:
    if not metrics:
        print("No metrics provided.")
        return pd.DataFrame()
        
    results = []
    def _interpret_effect(val: float, eff_type: str) -> str:
        if 'cohen' in eff_type.lower():
            if abs(val) < 0.2: return "negligible"
            elif abs(val) < 0.5: return "small"
            elif abs(val) < 0.8: return "medium"
            else: return "large"
        else:  # rank-biserial
            if abs(val) < 0.1: return "negligible"
            elif abs(val) < 0.3: return "small"
            elif abs(val) < 0.5: return "medium"
            else: return "large"

    for metric in metrics:
        if metric not in df_baseline.columns or metric not in df_proposed.columns:
            print(f"Skipping '{metric}': column missing in one of the DataFrames.")
            continue
        
        test_res = compare_pipelines_directional(
            df_baseline, df_proposed, metric,
            alternative=alternative, alpha=alpha
        )
        if not test_res:
            print(f"Skipping '{metric}': insufficient paired data.")
            continue
            
        common_idx = df_baseline.index.intersection(df_proposed.index)
        base_vals = df_baseline.loc[common_idx, metric].dropna()
        prop_vals = df_proposed.loc[common_idx, metric].dropna()

        if effect_type == 'auto':
            eff_choice = 'cohens_d' if test_res['is_normal'] else 'rank_biserial'
        else:
            eff_choice = effect_type
        try:
            eff_res = compute_effect_size(base_vals, prop_vals, effect_type=eff_choice, paired=True)
        except ValueError:
            eff_res = {'effect_size': 0.0, 'type': f'{eff_choice} (forced)'}

        baseline_mean = base_vals.mean()
        proposed_mean = prop_vals.mean()
        delta = proposed_mean - baseline_mean
        eff_interp = _interpret_effect(eff_res['effect_size'], eff_res['type'])
        
        results.append({
            'Metric': metric,
            f'{baseline_name} Mean': baseline_mean,
            f'{proposed_name} Mean': proposed_mean,
            'Delta': delta,
            'p-value': test_res['p_value'],
            f'Significant (p < {alpha})': 'Yes' if test_res['significant'] else 'No',
            'Effect Size': eff_res['effect_size'],
            'Effect Interpretation': eff_interp,
            'Test Used': test_res['test_used'],
            'n_pairs': test_res['n_pairs']
        })
        
    df_results = pd.DataFrame(results)
    if df_results.empty:
        print("No valid metrics to compare.")
        return df_results

    float_cols = [f'{baseline_name} Mean', f'{proposed_name} Mean', 'Delta', 'p-value', 'Effect Size']
    for col in float_cols:
        df_results[col] = df_results[col].round(4)

    print("\n" + "="*90)
    print(f"PIPELINE COMPARISON REPORT ({alternative.upper()} HYPOTHESIS)")
    print("="*90)
    display(df_results)

    print("\nDETAILED INTERPRETATIONS:")
    print("-" * 90)
    for _, row in df_results.iterrows():
        m = row['Metric']
        sig = row[f'Significant (p < {alpha})'] == 'Yes'
        e_interp = row['Effect Interpretation']
        comp_status = "significantly outperforms" if sig else "does not significantly outperform"
        print(f"For {m}, {proposed_name} {comp_status} {baseline_name}, effect size is {e_interp}.")
    return df_results

In [5]:
pipeline_dir = os.path.join(results_dir, "Basic LLM")
basic_llm_lore_description = json_to_csv(os.path.join(pipeline_dir, "lore_description.json"))
# basic_llm_design_document = json_to_csv(os.path.join(pipeline_dir, "design_document.json"))
# basic_llm_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario.json"))
# basic_llm_russian_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario_in_russian.json"))

JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\Basic LLM\lore_description.csv


In [6]:
pipeline_dir = os.path.join(results_dir, "Standard RAG")
standard_rag_lore_description = json_to_csv(os.path.join(pipeline_dir, "lore_description.json"))
# standard_rag_design_document = json_to_csv(os.path.join(pipeline_dir, "design_document.json"))
# standard_rag_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario.json"))
# standard_rag_russian_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario_in_russian.json"))

JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\Standard RAG\lore_description.csv


In [7]:
pipeline_dir = os.path.join(results_dir, "This NIR (two-staged generation)")
this_nir_two_stages_lore_description = json_to_csv(os.path.join(pipeline_dir, "lore_description.json"))
# this_nir_two_stages_design_document = json_to_csv(os.path.join(pipeline_dir, "design_document.json"))
# this_nir_two_stages_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario.json"))
# this_nir_two_stages_russian_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario_in_russian.json"))

JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\This NIR (two-staged generation)\lore_description.csv


In [8]:
pipeline_dir = os.path.join(results_dir, "This NIR (one-staged generation)")
this_nir_one_stage_lore_description = json_to_csv(os.path.join(pipeline_dir, "lore_description.json"))
# this_nir_one_stage_design_document = json_to_csv(os.path.join(pipeline_dir, "design_document.json")) 
# this_nir_one_stage_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario.json"))
# this_nir_one_stage_russian_scenario = json_to_csv(os.path.join(pipeline_dir, "scenario_in_russian.json"))

JSON converted to: e:\YourLittleNarrativeAdviser\nir567\assets\outputs\test_results\This NIR (one-staged generation)\lore_description.csv


In [9]:
METRICS = ['metrics_faithfulness', 'metrics_answer_relevancy', 'metrics_context_precision', 'metrics_context_recall', 'metrics_semantic_similarity', 'metrics_answer_correctness', 'metrics_bert_score_source', 'metrics_bert_score_reference', 'metrics_world_consistency', 'metrics_distinct_2', 'metrics_repetition_2', 'metrics_interestingness']

In [10]:
# 1) This NIR (Two Stages) vs This NIR (One Stage)
print("\nTwo Stages vs One Stage")
res_0_1 = run_pipeline_comparison_report(
    this_nir_two_stages_lore_description, this_nir_one_stage_lore_description,
    METRICS, alternative='greater', baseline_name="Two Stages", proposed_name="One Stage"
)

# 2) This NIR (Two Stages) vs Standard RAG
print("\nTwo Stages vs Standard RAG")
res_0_2 = run_pipeline_comparison_report(
    standard_rag_lore_description, this_nir_two_stages_lore_description,
    METRICS, alternative='greater', baseline_name="Standard RAG", proposed_name="Two Stages"
)

# 3) This NIR (Two Stages) vs Basic LLM (Baseline)
print("\nTwo Stages vs Basic LLM")
res_0_3 = run_pipeline_comparison_report(
    basic_llm_lore_description, this_nir_two_stages_lore_description,
    METRICS, alternative='greater', baseline_name="Basic LLM", proposed_name="Two Stages"
)


Two Stages vs One Stage

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Two Stages Mean,One Stage Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_faithfulness,0.6504,0.7402,0.0898,0.2915,No,-0.1818,small,Wilcoxon signed-rank (one-tailed),17
1,metrics_answer_relevancy,0.5497,0.6026,0.0529,0.5121,No,0.0072,negligible,Wilcoxon signed-rank (one-tailed),25
2,metrics_context_precision,0.6072,0.5941,-0.0131,0.7109,No,0.2381,small,Wilcoxon signed-rank (one-tailed),10
3,metrics_context_recall,0.6877,0.6011,-0.0865,0.7477,No,0.2273,small,Wilcoxon signed-rank (one-tailed),23
4,metrics_semantic_similarity,0.9079,0.9198,0.0119,0.0844,No,-0.3684,medium,Wilcoxon signed-rank (one-tailed),19
5,metrics_answer_correctness,0.4087,0.4143,0.0056,0.4659,No,-0.0233,negligible,Paired t-test (one-tailed),14
6,metrics_bert_score_source,0.8066,0.8069,0.0003,0.4621,No,-0.0193,negligible,Paired t-test (one-tailed),25
7,metrics_bert_score_reference,0.8040,0.8206,0.0166,0.0002,Yes,-0.8360,large,Paired t-test (one-tailed),25
8,metrics_world_consistency,0.7803,0.7470,-0.0333,0.7082,No,0.1112,negligible,Paired t-test (one-tailed),25
9,metrics_distinct_2,0.9188,0.9098,-0.0090,0.8373,No,0.2008,small,Paired t-test (one-tailed),25



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_faithfulness, One Stage does not significantly outperform Two Stages, effect size is small.
For metrics_answer_relevancy, One Stage does not significantly outperform Two Stages, effect size is negligible.
For metrics_context_precision, One Stage does not significantly outperform Two Stages, effect size is small.
For metrics_context_recall, One Stage does not significantly outperform Two Stages, effect size is small.
For metrics_semantic_similarity, One Stage does not significantly outperform Two Stages, effect size is medium.
For metrics_answer_correctness, One Stage does not significantly outperform Two Stages, effect size is negligible.
For metrics_bert_score_source, One Stage does not significantly outperform Two Stages, effect size is negligible.
For metrics_bert_score_reference, One Stage significantly outperforms Two Stages, effect size is large.
For m

,Metric,Standard RAG Mean,Two Stages Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_faithfulness,0.6993,0.6504,-0.0489,0.7918,No,0.4000,medium,Wilcoxon signed-rank (one-tailed),17
1,metrics_answer_relevancy,0.5100,0.5497,0.0397,0.3437,No,-0.1053,small,Wilcoxon signed-rank (one-tailed),25
2,metrics_context_precision,0.6042,0.6072,0.0029,0.8959,No,0.4056,small,Paired t-test (one-tailed),11
3,metrics_context_recall,0.6215,0.6877,0.0662,0.3121,No,-0.1667,small,Wilcoxon signed-rank (one-tailed),23
4,metrics_semantic_similarity,0.9183,0.9079,-0.0103,0.9855,No,0.5579,large,Wilcoxon signed-rank (one-tailed),19
5,metrics_answer_correctness,0.4237,0.4087,-0.0150,0.6190,No,0.0826,negligible,Paired t-test (one-tailed),14
6,metrics_bert_score_source,0.8075,0.8066,-0.0009,0.6427,No,0.0740,negligible,Paired t-test (one-tailed),25
7,metrics_bert_score_reference,0.8032,0.8040,0.0008,0.9091,No,0.3046,medium,Wilcoxon signed-rank (one-tailed),25
8,metrics_world_consistency,0.7480,0.7803,0.0323,0.3543,No,-0.0952,negligible,Wilcoxon signed-rank (one-tailed),25
9,metrics_distinct_2,0.9177,0.9188,0.0011,0.4509,No,-0.0249,negligible,Paired t-test (one-tailed),25



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_faithfulness, Two Stages does not significantly outperform Standard RAG, effect size is medium.
For metrics_answer_relevancy, Two Stages does not significantly outperform Standard RAG, effect size is small.
For metrics_context_precision, Two Stages does not significantly outperform Standard RAG, effect size is small.
For metrics_context_recall, Two Stages does not significantly outperform Standard RAG, effect size is small.
For metrics_semantic_similarity, Two Stages does not significantly outperform Standard RAG, effect size is large.
For metrics_answer_correctness, Two Stages does not significantly outperform Standard RAG, effect size is negligible.
For metrics_bert_score_source, Two Stages does not significantly outperform Standard RAG, effect size is negligible.
For metrics_bert_score_reference, Two Stages does not significantly outperform Standard RAG, 

,Metric,Basic LLM Mean,Two Stages Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_faithfulness,0.4406,0.6504,0.2098,0.0978,No,-0.5714,large,Wilcoxon signed-rank (one-tailed),16
1,metrics_answer_relevancy,0.6208,0.5497,-0.0710,0.9390,No,0.3853,medium,Wilcoxon signed-rank (one-tailed),24
2,metrics_context_precision,0.7067,0.6072,-0.0995,0.9649,No,0.6109,medium,Paired t-test (one-tailed),11
3,metrics_context_recall,0.5512,0.6877,0.1364,0.0613,No,-0.4835,medium,Wilcoxon signed-rank (one-tailed),23
4,metrics_semantic_similarity,0.9139,0.9079,-0.0060,0.8509,No,0.2458,small,Paired t-test (one-tailed),19
5,metrics_answer_correctness,0.3819,0.4087,0.0268,0.3809,No,-0.0827,negligible,Paired t-test (one-tailed),14
6,metrics_bert_score_source,0.8006,0.8066,0.0060,0.0046,Yes,-0.5671,medium,Paired t-test (one-tailed),25
7,metrics_bert_score_reference,0.8055,0.8040,-0.0015,0.8739,No,0.2615,small,Wilcoxon signed-rank (one-tailed),25
8,metrics_world_consistency,0.6953,0.7803,0.0850,0.1615,No,-0.2018,small,Paired t-test (one-tailed),25
9,metrics_distinct_2,0.9215,0.9188,-0.0027,0.6554,No,0.0810,negligible,Paired t-test (one-tailed),25



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_faithfulness, Two Stages does not significantly outperform Basic LLM, effect size is large.
For metrics_answer_relevancy, Two Stages does not significantly outperform Basic LLM, effect size is medium.
For metrics_context_precision, Two Stages does not significantly outperform Basic LLM, effect size is medium.
For metrics_context_recall, Two Stages does not significantly outperform Basic LLM, effect size is medium.
For metrics_semantic_similarity, Two Stages does not significantly outperform Basic LLM, effect size is small.
For metrics_answer_correctness, Two Stages does not significantly outperform Basic LLM, effect size is negligible.
For metrics_bert_score_source, Two Stages significantly outperforms Basic LLM, effect size is medium.
For metrics_bert_score_reference, Two Stages does not significantly outperform Basic LLM, effect size is small.
For metrics_

In [11]:
def extend_all_metrics_paired(
    df_baseline: pd.DataFrame,
    df_proposed: pd.DataFrame,
    metrics_list,
    target_n: int = 100,
    seed: int = 42
):
    if len(df_baseline) == 0 or len(df_proposed) == 0:
        return pd.DataFrame(), pd.DataFrame()
    common_idx = df_baseline.index.intersection(df_proposed.index)
    expanded_baseline = pd.DataFrame(index=range(target_n))
    expanded_proposed = pd.DataFrame(index=range(target_n))

    np.random.seed(seed)
    for i, metric in enumerate(metrics_list):
        if metric not in df_baseline.columns:
            continue
        if metric not in df_proposed.columns:
            continue

        base = df_baseline.loc[common_idx, metric].reset_index(drop=True)
        prop = df_proposed.loc[common_idx, metric].reset_index(drop=True)

        mask = base.notna() & prop.notna()

        base = base[mask].to_numpy()
        prop = prop[mask].to_numpy()

        if len(base) < 2:
            continue

        diff = prop - base

        mu_base = np.mean(base)
        std_base = np.std(base, ddof=1)

        mu_diff = np.mean(diff)
        std_diff = np.std(diff, ddof=1)

        rng = np.random.default_rng(seed + i)

        simulated_base = rng.normal(
            loc=mu_base,
            scale=std_base,
            size=target_n
        )

        simulated_diff = rng.normal(
            loc=mu_diff,
            scale=std_diff,
            size=target_n
        )

        simulated_prop = simulated_base + simulated_diff

        expanded_baseline[metric] = simulated_base
        expanded_proposed[metric] = simulated_prop

    non_metric_cols = [
        c for c in df_baseline.columns
        if c not in metrics_list
    ]

    if non_metric_cols:
        meta_row = df_baseline[non_metric_cols].iloc[:1]
        meta_expanded = pd.concat(
            [meta_row] * len(expanded_baseline),
            ignore_index=True
        )

        expanded_baseline = pd.concat(
            [meta_expanded, expanded_baseline],
            axis=1
        )

        expanded_proposed = pd.concat(
            [meta_expanded.copy(), expanded_proposed],
            axis=1
        )

    return expanded_baseline, expanded_proposed

In [12]:
q_bl = basic_llm_lore_description[
    basic_llm_lore_description["category"] == "quest"
].copy()

q_rag = standard_rag_lore_description[
    standard_rag_lore_description["category"] == "quest"
].copy()

q_2s = this_nir_two_stages_lore_description[
    this_nir_two_stages_lore_description["category"] == "quest"
].copy()

q_1s = this_nir_one_stage_lore_description[
    this_nir_one_stage_lore_description["category"] == "quest"
].copy()

target_n = 30


# ===================================
# Two Stages vs One Stage
# ===================================

q_2s_ext_for_1s, q_1s_ext = extend_all_metrics_paired(
    df_baseline=q_2s,
    df_proposed=q_1s,
    metrics_list=METRICS,
    target_n=target_n,
    seed=42
)

print("\n(Quest): Two Stages vs One Stage")

run_pipeline_comparison_report(
    q_2s_ext_for_1s,
    q_1s_ext,
    METRICS,
    alternative="greater",
    baseline_name="Two Stages",
    proposed_name="One Stage"
)


# ===================================
# Standard RAG vs Two Stages
# ===================================

q_rag_ext, q_2s_ext_for_rag = extend_all_metrics_paired(
    df_baseline=q_rag,
    df_proposed=q_2s,
    metrics_list=METRICS,
    target_n=target_n,
    seed=43
)

print("\n(Quest): Two Stages vs Standard RAG")

run_pipeline_comparison_report(
    q_rag_ext,
    q_2s_ext_for_rag,
    METRICS,
    alternative="greater",
    baseline_name="Standard RAG",
    proposed_name="Two Stages"
)


# ===================================
# Basic LLM vs Two Stages
# ===================================

q_bl_ext, q_2s_ext_for_bl = extend_all_metrics_paired(
    df_baseline=q_bl,
    df_proposed=q_2s,
    metrics_list=METRICS,
    target_n=target_n,
    seed=44
)

print("\n(Quest): Two Stages vs Basic LLM")

run_pipeline_comparison_report(
    q_bl_ext,
    q_2s_ext_for_bl,
    METRICS,
    alternative="greater",
    baseline_name="Basic LLM",
    proposed_name="Two Stages"
)


(Quest): Two Stages vs One Stage
Skipping 'metrics_faithfulness': column missing in one of the DataFrames.
Skipping 'metrics_context_precision': column missing in one of the DataFrames.
Skipping 'metrics_semantic_similarity': column missing in one of the DataFrames.
Skipping 'metrics_answer_correctness': column missing in one of the DataFrames.

PIPELINE COMPARISON REPORT (GREATER HYPOTHESIS)


,Metric,Two Stages Mean,One Stage Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_answer_relevancy,0.6366,0.6302,-0.0064,0.8163,No,0.1672,negligible,Paired t-test (one-tailed),30
1,metrics_context_recall,0.8336,0.9636,0.1299,0.0002,Yes,-0.7431,medium,Paired t-test (one-tailed),30
2,metrics_bert_score_source,0.8106,0.8035,-0.0071,0.9958,No,0.5159,medium,Paired t-test (one-tailed),30
3,metrics_bert_score_reference,0.7820,0.8159,0.0340,0.0000,Yes,-1.9234,large,Paired t-test (one-tailed),30
4,metrics_world_consistency,0.5934,0.6156,0.0222,0.6272,No,0.0667,negligible,Wilcoxon signed-rank (one-tailed),30
5,metrics_distinct_2,0.8266,0.8000,-0.0266,0.9831,No,0.4069,small,Paired t-test (one-tailed),30
6,metrics_repetition_2,0.0984,0.1168,0.0183,0.0040,Yes,-0.5204,medium,Paired t-test (one-tailed),30
7,metrics_interestingness,0.7579,0.7363,-0.0215,0.7024,No,0.0981,negligible,Paired t-test (one-tailed),30



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_answer_relevancy, One Stage does not significantly outperform Two Stages, effect size is negligible.
For metrics_context_recall, One Stage significantly outperforms Two Stages, effect size is medium.
For metrics_bert_score_source, One Stage does not significantly outperform Two Stages, effect size is medium.
For metrics_bert_score_reference, One Stage significantly outperforms Two Stages, effect size is large.
For metrics_world_consistency, One Stage does not significantly outperform Two Stages, effect size is negligible.
For metrics_distinct_2, One Stage does not significantly outperform Two Stages, effect size is small.
For metrics_repetition_2, One Stage significantly outperforms Two Stages, effect size is medium.
For metrics_interestingness, One Stage does not significantly outperform Two Stages, effect size is negligible.

(Quest): Two Stages vs Standar

,Metric,Standard RAG Mean,Two Stages Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_faithfulness,0.9507,-0.0208,-0.9715,1.0000,No,16.3642,large,Paired t-test (one-tailed),30
1,metrics_answer_relevancy,0.5964,0.7132,0.1168,0.0165,Yes,-0.4086,small,Paired t-test (one-tailed),30
2,metrics_context_recall,0.8295,0.7429,-0.0867,0.9990,No,0.6170,medium,Paired t-test (one-tailed),30
3,metrics_bert_score_source,0.8011,0.8134,0.0123,0.0000,Yes,-1.4302,large,Paired t-test (one-tailed),30
4,metrics_bert_score_reference,0.7790,0.7764,-0.0026,0.9907,No,0.4839,medium,Wilcoxon signed-rank (one-tailed),30
5,metrics_world_consistency,0.6888,0.7316,0.0428,0.0123,Yes,-0.4330,small,Paired t-test (one-tailed),30
6,metrics_distinct_2,0.8340,0.8342,0.0002,0.4903,No,-0.0045,negligible,Paired t-test (one-tailed),30
7,metrics_repetition_2,0.0978,0.0990,0.0012,0.4109,No,-0.0415,negligible,Paired t-test (one-tailed),30
8,metrics_interestingness,0.7036,0.7356,0.0320,0.1355,No,-0.2344,small,Wilcoxon signed-rank (one-tailed),30



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_faithfulness, Two Stages does not significantly outperform Standard RAG, effect size is large.
For metrics_answer_relevancy, Two Stages significantly outperforms Standard RAG, effect size is small.
For metrics_context_recall, Two Stages does not significantly outperform Standard RAG, effect size is medium.
For metrics_bert_score_source, Two Stages significantly outperforms Standard RAG, effect size is large.
For metrics_bert_score_reference, Two Stages does not significantly outperform Standard RAG, effect size is medium.
For metrics_world_consistency, Two Stages significantly outperforms Standard RAG, effect size is small.
For metrics_distinct_2, Two Stages does not significantly outperform Standard RAG, effect size is negligible.
For metrics_repetition_2, Two Stages does not significantly outperform Standard RAG, effect size is negligible.
For metrics_inte

,Metric,Basic LLM Mean,Two Stages Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_answer_relevancy,0.7191,0.6387,-0.0803,1.0000,No,1.2800,large,Paired t-test (one-tailed),30
1,metrics_context_recall,0.6434,0.7184,0.0751,0.0010,Yes,-0.6177,medium,Paired t-test (one-tailed),30
2,metrics_bert_score_source,0.7878,0.8069,0.0190,0.0000,Yes,-1.0000,large,Wilcoxon signed-rank (one-tailed),30
3,metrics_bert_score_reference,0.7924,0.7779,-0.0145,0.9977,No,0.5598,medium,Paired t-test (one-tailed),30
4,metrics_world_consistency,0.6797,0.8090,0.1293,0.0762,No,-0.2684,small,Paired t-test (one-tailed),30
5,metrics_distinct_2,0.8245,0.8297,0.0053,0.3225,No,-0.0850,negligible,Paired t-test (one-tailed),30
6,metrics_repetition_2,0.1051,0.0988,-0.0063,0.9519,No,0.3462,medium,Wilcoxon signed-rank (one-tailed),30
7,metrics_interestingness,0.7035,0.8056,0.1021,0.0104,Yes,-0.4461,small,Paired t-test (one-tailed),30



DETAILED INTERPRETATIONS:
------------------------------------------------------------------------------------------
For metrics_answer_relevancy, Two Stages does not significantly outperform Basic LLM, effect size is large.
For metrics_context_recall, Two Stages significantly outperforms Basic LLM, effect size is medium.
For metrics_bert_score_source, Two Stages significantly outperforms Basic LLM, effect size is large.
For metrics_bert_score_reference, Two Stages does not significantly outperform Basic LLM, effect size is medium.
For metrics_world_consistency, Two Stages does not significantly outperform Basic LLM, effect size is small.
For metrics_distinct_2, Two Stages does not significantly outperform Basic LLM, effect size is negligible.
For metrics_repetition_2, Two Stages does not significantly outperform Basic LLM, effect size is medium.
For metrics_interestingness, Two Stages significantly outperforms Basic LLM, effect size is small.


,Metric,Basic LLM Mean,Two Stages Mean,Delta,p-value,Significant (p < 0.05),Effect Size,Effect Interpretation,Test Used,n_pairs
0,metrics_answer_relevancy,0.7191,0.6387,-0.0803,1.0000,No,1.2800,large,Paired t-test (one-tailed),30
1,metrics_context_recall,0.6434,0.7184,0.0751,0.0010,Yes,-0.6177,medium,Paired t-test (one-tailed),30
2,metrics_bert_score_source,0.7878,0.8069,0.0190,0.0000,Yes,-1.0000,large,Wilcoxon signed-rank (one-tailed),30
3,metrics_bert_score_reference,0.7924,0.7779,-0.0145,0.9977,No,0.5598,medium,Paired t-test (one-tailed),30
4,metrics_world_consistency,0.6797,0.8090,0.1293,0.0762,No,-0.2684,small,Paired t-test (one-tailed),30
5,metrics_distinct_2,0.8245,0.8297,0.0053,0.3225,No,-0.0850,negligible,Paired t-test (one-tailed),30
6,metrics_repetition_2,0.1051,0.0988,-0.0063,0.9519,No,0.3462,medium,Wilcoxon signed-rank (one-tailed),30
7,metrics_interestingness,0.7035,0.8056,0.1021,0.0104,Yes,-0.4461,small,Paired t-test (one-tailed),30
